# Seamless SAT Frames | Thermal infrared interpolation

Run the same pipeline as the dashboard on a real 30-minute Hurricane Milton sequence. The notebook loads aligned thermal observations, predicts the missing frames, compares both methods against the observed midpoint, and exports PNG and HDF5 artifacts.

**Fast path:** run sections 0-5. Sections 6-7 show how to acquire new observations, train a checkpoint and run the web app.

| Frame | Meaning | Milton scan start (UTC) |
|---|---|---|
| First | Endpoint provided to both methods | 9 Oct 2024, 18:01:17.4 |
| Middle | Observed reference held out from inference | 9 Oct 2024, 18:16:17.4 |
| Last | Endpoint provided to both methods | 9 Oct 2024, 18:31:17.4 |

[NOAA source archive](https://noaa-goes16.s3.amazonaws.com/?list-type=2&prefix=ABI-L2-CMIPC%2F2024%2F283%2F18%2F)

## 0. Environment and project files

Open this notebook from the project root in VS Code or Jupyter and select the backend virtual environment as the kernel. If the environment needs kernel support, run `python -m pip install ipykernel` using that environment. For Colab, upload a ZIP of the project source (include `dashboard/backend/sample`, exclude `.git`, `.venv`, `node_modules`, `dist`, and raw data), and run the setup cell. CPU works for inference. The optional training section can use a Colab GPU.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
from zipfile import ZipFile

start_dir = Path.cwd().resolve()
in_colab = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

def locate_project(base):
    for candidate in (base, *base.parents, *base.glob("*")):
        if (candidate / "dashboard/backend/server.py").is_file():
            return candidate.resolve()
    return None

repo = locate_project(Path(os.environ.get("SAT_PROJECT_ROOT", start_dir)).resolve())
if repo is None and in_colab:
    from google.colab import files
    uploaded = files.upload()
    archive = next((Path(name) for name in uploaded if name.lower().endswith(".zip")), None)
    if archive is None:
        raise FileNotFoundError("Upload a ZIP of this project source.")
    extracted = Path("/content/sat-project")
    with ZipFile(archive) as bundle:
        bundle.extractall(extracted)
    repo = locate_project(extracted)
if repo is None:
    raise FileNotFoundError("Open the notebook from the project root or set SAT_PROJECT_ROOT.")

backend = repo / "dashboard/backend"
modules = ("flask", "flask_cors", "cv2", "numpy", "h5py", "torch", "piq")
if any(importlib.util.find_spec(name) is None for name in modules):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(backend / "requirements.txt")], check=True)

import torch
print("Project:", repo)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Load a matched triplet

The bundled Milton images use one fixed 384 x 384 crop of GOES-16 ABI Channel 13 and a shared 180-330 K display scale. To try your own sequence, change the three paths and set the actual endpoint gap. Use a matching projection, crop, and thermal band for every frame. Leave the midpoint path as `None` for interpolation without evaluation.

In [ ]:
import cv2
import numpy as np

sample = backend / "sample/milton"
first_path = sample / "frame_000.png"
middle_path = sample / "frame_001.png"
last_path = sample / "frame_002.png"
gap_minutes = 30

def read_gray(path):
    frame = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if frame is None:
        raise ValueError(f"Could not read {path}")
    return frame

first, last = read_gray(first_path), read_gray(last_path)
assert first.shape == last.shape, "Endpoint grids must match."
assert not np.array_equal(first, last), "The two endpoints contain the same pixels."
if middle_path is not None:
    assert read_gray(middle_path).shape == first.shape, "The observed midpoint grid must match."
print("Grid:", first.shape, "| Endpoint gap:", gap_minutes, "minutes")

## 2. Run the project backend

Use Flask's in-process client, so the notebook exercises the same upload route as the web dashboard. The linear method weights the original endpoints at 25%, 50%, and 75%. U-Net uses a learned midpoint and recursive quarter passes. The observed middle frame is used only for scoring and visualization. U-Net scores are exploratory because the starter checkpoint uses a small training set.

In [ ]:
from contextlib import contextmanager
from io import BytesIO
from uuid import uuid4
import shutil

if str(backend) not in sys.path:
    sys.path.insert(0, str(backend))
working_dir = Path.cwd()
try:
    import server
finally:
    os.chdir(working_dir)

output_base = Path(os.environ.get("SAT_NOTEBOOK_OUTPUT", repo / "notebook-output")).resolve()
output = output_base / ("run-" + uuid4().hex[:8])
output.mkdir(parents=True)
client = server.app.test_client()

@contextmanager
def isolated_run(directory):
    previous = Path.cwd()
    directory.mkdir(parents=True, exist_ok=True)
    os.chdir(directory)
    try:
        yield
    finally:
        os.chdir(previous)

def evaluate_method(method):
    fields = {
        "frame1": (BytesIO(first_path.read_bytes()), first_path.name),
        "frame3": (BytesIO(last_path.read_bytes()), last_path.name),
        "resolution": "7.5min",
        "endpoint_gap": str(gap_minutes),
        "method": method,
    }
    if middle_path is not None:
        fields["midpoint"] = (BytesIO(middle_path.read_bytes()), middle_path.name)
    folder = output / method
    with isolated_run(folder):
        reply = client.post("/interpolate", data=fields)
        if reply.status_code != 200:
            raise RuntimeError(reply.get_json() or reply.status)
        export = client.get("/download_h5")
        if export.status_code != 200:
            raise RuntimeError(export.get_json() or export.status)
        (folder / "prediction.h5").write_bytes(export.data)
        export.close()
        (folder / "data/latest_prediction.h5").unlink()
        (folder / "data").rmdir()
    return reply.get_json()

runs = {"linear": evaluate_method("linear")}
if client.get("/api/health").get_json()["u_net_available"]:
    runs["u-net"] = evaluate_method("u-net")
else:
    print("U-Net weights unavailable. Continuing with the linear blend.")
print("Completed:", ", ".join(runs))

## 3. Inspect the frames and diagnose errors

Compare the observed midpoint with the predictions. The endpoint-change layer shows absolute intensity differences over the 30-minute gap. The midpoint error layer measures absolute error against the actual middle observation on the working grid.

In [ ]:
from IPython.display import HTML, display
from html import escape

def show(images):
    cards = []
    for label, src in images:
        if src:
            cards.append(f'<figure style="margin:0;width:200px"><img src="{src}" '
                         f'style="width:100%" alt="{escape(label, quote=True)}">'
                         f'<figcaption>{escape(label)}</figcaption></figure>')
    display(HTML('<div style="display:flex;gap:14px;flex-wrap:wrap">' + "".join(cards) + "</div>"))

linear = runs["linear"]
show([
    ("First observation", linear["start_image"]),
    ("Observed midpoint", linear["ground_truth"]),
    ("Linear prediction", linear["base_image"]),
    ("U-Net prediction", runs.get("u-net", {}).get("base_image")),
    ("Last observation", linear["end_image"]),
])
show([
    ("Endpoint change", linear["endpoint_change_image"]),
    ("Linear absolute midpoint error", linear["observed_error_image"]),
    ("U-Net absolute midpoint error", runs.get("u-net", {}).get("observed_error_image")),
])

## 4. Score against the held-out observation

Higher SSIM, PSNR, and FSIM are better. Lower MSE is better. Each score uses the same 8-bit working grid and the same observed midpoint. Treat this run as one case study. A model comparison needs multiple unseen sequences.

In [ ]:
metrics = [("ssim", "SSIM"), ("psnr", "PSNR (dB)"), ("mse", "MSE"), ("fsim", "FSIM")]
if middle_path is None:
    print("Provide an observed midpoint to calculate scores.")
else:
    headers = "".join(f"<th>{name}</th>" for name in runs)
    rows = []
    for key, title in metrics:
        values = "".join(f"<td>{result['metrics'].get(key) or 'Unavailable'}</td>" for result in runs.values())
        rows.append(f"<tr><th>{title}</th>{values}</tr>")
    display(HTML("<table><thead><tr><th>Metric</th>" + headers + "</tr></thead><tbody>"
                 + "".join(rows) + "</tbody></table>"))

## 5. Export PNG and HDF5

The HDF5 files contain the generated quarter, midpoint, and three-quarter frames, along with the method and offsets in minutes. The PNG exports use the same working grid as the HDF5 datasets.

In [ ]:
from base64 import b64decode
import h5py

for method, result in runs.items():
    folder = output / method
    quarter, third_quarter = result["extra_frames"]
    for filename, data in (("quarter.png", quarter), ("midpoint.png", result["base_image"]),
                           ("third_quarter.png", third_quarter)):
        (folder / filename).write_bytes(b64decode(data.split(",", 1)[1]))
    with h5py.File(folder / "prediction.h5", "r") as export:
        offsets = [float(export[key].attrs["offset_minutes"])
                   for key in ("first_quarter", "interpolated_frame", "third_quarter")]
        assert offsets == [gap_minutes * fraction for fraction in (0.25, 0.5, 0.75)]
        assert export.attrs["engine"] == method
        print(method, "| offsets:", offsets, "| grid:", export["interpolated_frame"].shape)
print("Artifacts:", output)

## 6. Optional | Download calibrated GOES data and train

Enable the next cell to reproduce the starter training recipe. It downloads two GOES-18 dates distinct from Milton, crops each sequence on a fixed grid, maps all observations to the same scale, then trains the reference residual U-Net. The training command selects CUDA when available. The new checkpoint lands in this notebook run's output directory. To test it in the dashboard, place it at `dashboard/backend/models/satellite_interpolator.pth` and restart Flask.

[NOAA GOES open archive](https://registry.opendata.aws/noaa-goes/) | [Reference U-Net](https://github.com/pradhan-not-found/ISRO/blob/main/backend/model.py)

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    def run_script(script, *args):
        subprocess.run([sys.executable, str(backend / script), *map(str, args)], cwd=backend, check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "xarray", "netCDF4"], check=True)
    sequences = []
    for date in ("2024-01-02", "2024-01-04"):
        raw = output / "training" / date / "raw"
        processed = output / "training" / date / "processed"
        run_script("download_sample.py", "--date", date, "--hour", 11, "--count", 6, "--output", raw)
        run_script("preprocess.py", "--raw-dir", raw, "--output", processed)
        sequences.append(processed)
    checkpoint = output / "satellite_interpolator.pth"
    run_script("train.py", "--data-dir", *sequences, "--epochs", 4, "--steps", 80,
               "--patch-size", 64, "--seed", 42, "--output", checkpoint)
    print("Checkpoint:", checkpoint)

## 7. Run the web app

From the repository root, open two PowerShell terminals.

**Backend**
```powershell
cd dashboard/backend
python -m venv .venv
./.venv/Scripts/python.exe -m pip install -r requirements.txt
./.venv/Scripts/python.exe server.py
```

**Frontend**
```powershell
cd dashboard/frontend
npm.cmd ci
npm.cmd run dev
```

Open [localhost:5173](http://localhost:5173), select **Milton**, and run it with the default **Linear blend**. Select **U-Net (experimental)** to compare against the blend. For your own frames, choose **Upload** and fill the same three slots used in section 1.

The **Satellites** page is a separate viewer for current MOSDAC satellite, radar and forecast products. It does not feed imagery into the interpolation API automatically.